# Hartmann1D — the induced field of the Hartmann flow (1D) (2D + 3D)

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = (b \cdot \nabla) u$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data. Every case is **self-contained** (inline `Mailler`/`Pave` mesh, no
`.med` file) and records point probes (`Postraitement/Sondes`) into `.son`
files which are compared below against a closed-form solution.

This notebook covers the **`hartmann_1d`** validation form. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.


Both dimensions are run from a single shared data template in this form; the 3D mesh is converted with `Polyedriser dom`.
## What it validates

The **Hartmann problem** — a conducting fluid between two parallel plates with
a uniform transverse magnetic field — is the canonical MHD benchmark. The
module does **not** solve the momentum equation (no Lorentz force, model M2 is
deferred), so the *velocity* is not computed here: the exact Hartmann profile
is prescribed as data,

$$u(y) = U_0\,\frac{\cosh(\mathrm{Ha}) - \cosh(\mathrm{Ha}\,y/h)}{\cosh(\mathrm{Ha}) - 1},
\qquad u = (u(y), 0),$$

with no-slip at both plates ($y = \pm h$, $h = 0.5$) and Hartmann number
$\mathrm{Ha} = 5$. What *is* solved and validated here is the **induced
magnetic field**: the applied field $b_y = B_0 = 1$ along $y$ makes the M1
source $S_x = B_0\,u'(y)$, so the steady induction equation with **insulating
walls** (zero normal flux of $b_x$, i.e. zero wall current) reads

$$\eta\,b_x''(y) = -B_0\,u'(y) \quad\text{on } (-h,h),\qquad b_x'(\pm h)=0,$$

whose mean-free solution is the odd profile

$$b_x(y) = -\frac{B_0}{\eta}\left[\int_{-h}^{y} u \, \mathrm{d}y' - \frac12\int_{-h}^{h} u \, \mathrm{d}y'\right],$$

with $b_x(0) = 0$ and $b_x'(\pm h) = -(B_0/\eta)\,u(\pm h) = 0$. Starting from
$b = (0, B_0)$ the mean of $b_x$ is conserved (zero-flux walls and
$u(\pm h) = 0$), so the transient relaxes onto exactly that mean-free profile,
while $b_y = B_0$ is an exact invariant of the full system ($S_y = 0$).

One-dimensional physics on a 2D mesh: every field depends on $y$ only. This
exercises the multi-component transport ($b = (b_x, b_y)$, `nb_composantes 2`),
the M1 `induction_stretching` source driving $b_x$ through the applied $b_y$,
the diffusion operator, and the insulating (Neumann) walls.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 7b | M1 source $S=(b\cdot\nabla)u$ drives the induced field | $b_x \to$ analytic mean-free profile (duct core) |
| 6 | diffusion in divergence form, insulating walls $\partial_n b=0$ | odd profile, $b_x'(\pm h)=0$ |
| 7b | components transported independently, $b_y$ untouched ($S_y=0$) | $b_y \equiv B_0$ to machine precision |

> **Known limitation (walls).** The kernel velocity gradient that feeds the M1
> source is only first-order accurate in the two cells adjacent to a wall
> (measured on a linear velocity: its value is exactly halved there), which
> leaves a small, converging deviation of the induced field inside the
> Hartmann layer. The core probes are therefore compared **tightly** and the
> layer probes only against a **documented bound**; see the closing section and
> `docs/content/dev_corner/kernel_reuse.md`.

In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: analytic Hartmann reference and the `.son` probe reader

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import math

import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name, subdir=None):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    if subdir is None:
        subdir = "3D/coarse" if "_3d" in case.lower() else "coarse"
    path = f"{BUILD}/{subdir}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# --- analytic Hartmann reference (same constants as the dataset) -----------
HA, H, U0, B0, ETA = 5.0, 0.5, 0.2, 1.0, 0.05
_CH, _SH = math.cosh(HA), math.sinh(HA)

def u_hartmann(y):
    """Analytic Hartmann profile, no-slip at y = +/-H."""
    return U0 * (_CH - math.cosh(HA * y / H)) / (_CH - 1.0)

def _int_u(s):
    """Integral of u from -H to s."""
    return U0 / (_CH - 1.0) * ((s + H) * _CH - (H / HA) * (math.sinh(HA * s / H) + _SH))

_I_TOT = _int_u(H)                      # = integral of u over the gap
B_SCALE = (B0 / ETA) * _I_TOT / 2.0     # steady-state amplitude of b_x

def b_x_steady(y):
    """Mean-free steady induced field with insulating walls."""
    return -(B0 / ETA) * (_int_u(y) - _I_TOT / 2.0)

# probes sit exactly on cell centres: y_k = -0.5 + (k+1/2)/64, x = 0.125
CORE = [("SONDE_C1", -0.0546875), ("SONDE_C2", -0.0234375),
        ("SONDE_C3", 0.0078125), ("SONDE_C4", 0.0390625),
        ("SONDE_C5", 0.0703125)]
WALL = [("SONDE_W1", -0.3828125), ("SONDE_W2", 0.3828125)]

print("BUILD =", BUILD)
print(f"steady amplitude B_SCALE = {B_SCALE:.6f}")


## Run both dimensions

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
from string import Template
from trustutils import run

NB_PROCS = 2
run.reset()

# Mesh geometries and physics/post-processing blocks are the two dimensions
# of the same src template. The 3D case uses a distinct target data name
# and build subtree, so partition zones and probe files cannot collide.
MESH_BLOCKS = {
    2: r"""Mailler dom
{
    Pave pave
    {
        Origine              0. -0.5
        Nombre_de_Noeuds     $nx $ny
        Longueurs            1.  1.
    }
    {
        Bord gauche   X = 0.   -0.5 <= Y <= 0.5
        Bord droite   X = 1.   -0.5 <= Y <= 0.5
        Bord bas      Y = -0.5  0. <= X <= 1.
        Bord haut     Y = 0.5   0. <= X <= 1.
    }
}""",
    3: r"""Mailler dom
{
    Pave cube
    {
        Origine              0. -0.5 0.
        Nombre_de_Noeuds     $nx $ny $nz
        Longueurs            1.  1.  1.
    }
    {
        Bord gauche   X = 0.   -0.5 <= Y <= 0.5  0. <= Z <= 1.
        Bord droite   X = 1.   -0.5 <= Y <= 0.5  0. <= Z <= 1.
        Bord bas      Y = -0.5  0. <= X <= 1.  0. <= Z <= 1.
        Bord haut     Y = 0.5   0. <= X <= 1.  0. <= Z <= 1.
        Bord devant   Z = 0.    0. <= X <= 1.  -0.5 <= Y <= 0.5
        Bord derriere Z = 1.    0. <= X <= 1.  -0.5 <= Y <= 0.5
    }
}

Polyedriser dom""",
}
DIMENSION_BLOCKS = {
    2: {
        'boundary_block': r"""boundary_conditions
        {
            # insulating walls: zero normal flux on both components #
            gauche  Neumann_Paroi_adiabatique
            droite  Neumann_Paroi_adiabatique
            bas     Neumann_Paroi_adiabatique
            haut    Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      # probes exactly on cell centres: x = 0.125, y_k = -0.5 + (k+1/2)/64 #
      # core cells (|y| <= 0.07): tight comparison with the analytic profile #
      sonde_c1  grav magnetic_field0 Periode 1e-6 Points 1 0.125 -0.0546875
      sonde_c2  grav magnetic_field0 Periode 1e-6 Points 1 0.125 -0.0234375
      sonde_c3  grav magnetic_field0 Periode 1e-6 Points 1 0.125  0.0078125
      sonde_c4  grav magnetic_field0 Periode 1e-6 Points 1 0.125  0.0390625
      sonde_c5  grav magnetic_field0 Periode 1e-6 Points 1 0.125  0.0703125
      # Hartmann-layer cells (|y| ~ 0.38): bounded by the module's known  #
      # boundary limitation (see the notebook), not tight                #
      sonde_w1  grav magnetic_field0 Periode 1e-6 Points 1 0.125 -0.3828125
      sonde_w2  grav magnetic_field0 Periode 1e-6 Points 1 0.125  0.3828125
      # applied transverse field: exact invariant #
      sonde_by  grav magnetic_field1 Periode 1e-6 Points 1 0.125  0.0078125
    }
  }""",
        'velocity_line': r"""        vitesse_magnetique     Champ_Fonc_xyz dom 2 0.2*(cosh(5.)-cosh(5.*y/0.5))/(cosh(5.)-1.) 0.""",
    },
    3: {
        'boundary_block': r"""boundary_conditions
        {
            # insulating walls: zero normal flux on both components #
            gauche   Neumann_Paroi_adiabatique
            droite   Neumann_Paroi_adiabatique
            bas      Neumann_Paroi_adiabatique
            haut     Neumann_Paroi_adiabatique
            devant   Neumann_Paroi_adiabatique
            derriere Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      # probes exactly on cell centres: x = z = 0.125, y_k = -0.5 + (k+1/2)/64 #
      # core cells (|y| <= 0.07): tight comparison with the analytic profile #
      sonde_c1  grav magnetic_field0 Periode 1e-6 Points 1 0.125 -0.0546875 0.125
      sonde_c2  grav magnetic_field0 Periode 1e-6 Points 1 0.125 -0.0234375 0.125
      sonde_c3  grav magnetic_field0 Periode 1e-6 Points 1 0.125  0.0078125 0.125
      sonde_c4  grav magnetic_field0 Periode 1e-6 Points 1 0.125  0.0390625 0.125
      sonde_c5  grav magnetic_field0 Periode 1e-6 Points 1 0.125  0.0703125 0.125
      # Hartmann-layer cells (|y| ~ 0.38): documented bound, not tight #
      sonde_w1  grav magnetic_field0 Periode 1e-6 Points 1 0.125 -0.3828125 0.125
      sonde_w2  grav magnetic_field0 Periode 1e-6 Points 1 0.125  0.3828125 0.125
      # applied transverse field: exact invariant #
      sonde_by  grav magnetic_field1 Periode 1e-6 Points 1 0.125  0.0078125 0.125
    }
  }""",
        'velocity_line': r"""        vitesse_magnetique     Champ_Fonc_xyz dom 3 0.2*(cosh(5.)-cosh(5.*y/0.5))/(cosh(5.)-1.) 0. 0.""",
    },
}

MESHES_2D = {'coarse': {'nx': 5, 'ny': 65}, 'standard': {'nx': 5, 'ny': 129}, 'fine': {'nx': 5, 'ny': 257}}
MESHES_3D = {'coarse': {'nx': 5, 'ny': 65, 'nz': 5}, 'standard': {'nx': 5, 'ny': 129, 'nz': 5}, 'fine': {'nx': 5, 'ny': 257, 'nz': 5}}

def register_dimension(dim, meshes, prefix="", target_data=None, nb_procs=NB_PROCS, partition=True):
    registered = {}
    for level, mesh in meshes.items():
        dic = {"dim": dim,
               "mesh_block": Template(MESH_BLOCKS[dim]).substitute(**mesh),
               **DIMENSION_BLOCKS[dim]}
        target = f"{prefix}/{level}" if prefix else level
        kwargs = {"nbProcs": nb_procs, "excluNR": True}
        if target_data is not None:
            kwargs["targetData"] = target_data
        registered[level] = run.addCaseFromTemplate('hartmann_1d.data', target, dic=dic, **kwargs)
        if partition:
            registered[level].partition()
    return registered

cases = register_dimension(2, MESHES_2D)
cases_3d = register_dimension(3, MESHES_3D, prefix="3D", target_data='hartmann_1d_3d.data', nb_procs=NB_PROCS, partition=True)
case = cases["coarse"]
case_3d = cases_3d["coarse"]
run.printCases()
run.runCases()


## 2D checks


In [ ]:
# 1) the applied transverse field must remain exactly uniform (exact invariant)
t, by = read_son(case.dataFileName_, "SONDE_BY")
drift_by = float(np.max(np.abs(by - B0)))
print(f"b_y: b(0)={by[0]:.6f}  max|b_y - B0| = {drift_by:.2e}")
assert drift_by < 1e-10, f"applied transverse field drifted: {drift_by:.3e}"
print("OK: b_y = B0 is an exact invariant (S_y = 0).")


In [ ]:
# 2) the core induced field must match the analytic mean-free Hartmann profile
TOL_CORE = 1.5e-2    # max |b_x_sim - b_x_ref| / B_SCALE over the duct core
STATIONARY = 1e-3    # residual transient on the last recorded step

ys, sim, ref, errs, steps = [], [], [], [], []
for tag, y in CORE:
    t, bx = read_son(case.dataFileName_, tag)
    r = b_x_steady(y)
    e = abs(bx[-1] - r) / B_SCALE
    step = abs(bx[-1] - bx[-2]) / B_SCALE
    ys.append(y); sim.append(bx[-1]); ref.append(r); errs.append(e); steps.append(step)
    print(f"y={y:+.7f}  b_x(t_end)={bx[-1]:+.6f}  analytic={r:+.6f}  "
          f"err/B_SCALE={e:.2e}  last-step={step:.2e}")

err, last_step = max(errs), max(steps)
print(f"core: max |b_x - analytic| / B_SCALE = {err:.3e}   (last-step {last_step:.2e})")
assert err < TOL_CORE, f"core induced field off by {err:.3e} (> {TOL_CORE:g})"
assert last_step < STATIONARY, f"not converged to steady state: last step {last_step:.3e}"
print("OK: core induced field matches the analytic insulating-wall Hartmann profile.")


In [ ]:
# 3) the Hartmann-layer deviation is ODD and bounded (known boundary limitation)
TOL_WALL = 1.0e-1

wall_vals, wall_err = [], []
for tag, y in WALL:
    t, bx = read_son(case.dataFileName_, tag)
    r = b_x_steady(y)
    e = abs(bx[-1] - r) / B_SCALE
    wall_vals.append(bx[-1]); wall_err.append(e)
    print(f"y={y:+.7f}  b_x(t_end)={bx[-1]:+.6f}  analytic={r:+.6f}  err/B_SCALE={e:.2e}")

asym = abs(wall_vals[0] + wall_vals[1]) / B_SCALE
print(f"wall-layer: max err/B_SCALE = {max(wall_err):.2e}, odd symmetry |b(-y)+b(y)|/B_SCALE = {asym:.2e}")
assert asym < 1e-10, f"induced field lost its odd symmetry: {asym:.3e}"
assert max(wall_err) < TOL_WALL, f"wall-layer deviation {max(wall_err):.3e} exceeds the documented bound {TOL_WALL:g}"
print("OK: layer deviation is odd-symmetric and within the documented bound.")


In [ ]:
import matplotlib.pyplot as plt

yy = np.linspace(-H, H, 201)
plt.figure(figsize=(6, 4))
plt.plot(yy, [b_x_steady(y) for y in yy], "k-", label="analytic $b_x(y)$ (steady)")
plt.plot([y for _, y in CORE], sim, "o", label="simulated (core probes)")
plt.plot([y for _, y in WALL], wall_vals, "s", mfc="none",
         label="simulated (layer probes, known limitation)")
plt.axhline(0.0, color="grey", lw=0.5)
plt.xlabel("y [m]"); plt.ylabel("$b_x$ [T]")
plt.title("Hartmann 1D: induced field, insulating walls")
plt.legend(); plt.grid(True); plt.show()


## Known limitation: the velocity gradient at wall cells

The M1 source needs the cell gradient $\nabla u$ of the frozen velocity. The
kernel builds it (`Discret_Thyd::grad_u` → `grad_Champ_Face_PolyMAC_MPFA`) by
*averaging the face gradients* (`update_ge()`), and a boundary face only
receives a Dirichlet contribution — the frozen velocity field carries no
hydrodynamic wall condition here. The two cells adjacent to a wall therefore
receive a biased $\nabla u$. Isolating the source (run the case with $\eta=0$
and a velocity $u = a\,y$, which is exactly representable) makes the effect
readable: the implied $\mathrm{d}u/\mathrm{d}y$ is $0.150$ instead of $0.300$ at
the wall cells, and correct ($0.300$) in the interior.

Consequences, measured on this dataset (64 cells across the gap):

- the deviation $b_x^{sim} - b_x^{analytic}$ is an almost exact **linear mode
  in $y$** ($\approx 0.305\,y$), i.e. it vanishes at the centre and is largest
  at the walls: $\approx 1\%$ at $|y| = 0.05$, $\approx 9\%$ at $|y| = 0.49$;
- it is **convergent, first order** (halves when the mesh is refined);
- it is odd (the wall-cell bias is symmetric), so the mean and the odd
  symmetry of the profile are preserved exactly.

This is why the core probes are asserted tightly (a few $10^{-2}$ on a 64-cell
mesh) and the layer probes only against the bound $10^{-1}$: the bound is a
**regression guard on the known limitation**, and it can be tightened once the
boundary treatment of the frozen velocity's gradient is fixed.


## 2D mesh convergence

Shared refinement ladder on the **active direction** (every field depends on
$y$ only, so the transverse $x$ node count stays at its baseline value):
`coarse` = baseline `5×65`, `standard` = $y$-cells ×2 (`5×129`), `fine` = $y$-cells ×4
(`5×257`). The steady state is dt-independent (implicit Euler fixed point),
so the baseline `tmax`/`dt_max` are kept on every level.

The probes are re-snapped to refined cell centres, so the analytic reference
is evaluated at the **effective** probe position read from each `.son`
header. The expected dominant error is the documented wall-cell gradient bias
(see *Known limitation* below): first-order convergent — the scan measures
exactly that claim. $b_y = B_0$ must stay invariant at every level.


In [ ]:
def probe_xy(path):
    """Effective (x, y) of a moved point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2])
    raise RuntimeError(f"no probe position header in {path}")

def read_son_dir(subdir, case_name, name):
    """(time, values, (x_eff, y_eff)) of probe `name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float), probe_xy(path)

def level_errors(subdir, case_name):
    """(core_err, wall_err, by_drift) at final time, effective probe positions."""
    core = wall = drift = 0.0
    for tag, _ in CORE:
        _, bx, (_, y_eff) = read_son_dir(subdir, case_name, tag)
        core = max(core, abs(bx[-1] - b_x_steady(y_eff)) / B_SCALE)
    for tag, _ in WALL:
        _, bx, (_, y_eff) = read_son_dir(subdir, case_name, tag)
        wall = max(wall, abs(bx[-1] - b_x_steady(y_eff)) / B_SCALE)
    _, by, _ = read_son_dir(subdir, case_name, "SONDE_BY")
    drift = float(np.max(np.abs(by - B0)))
    return core, wall, drift

scan = {"coarse": ("coarse", case.dataFileName_),                       # the baseline mesh (coarse), in build/coarse/
        "standard": ("standard", cases["standard"].dataFileName_),
        "fine": ("fine", cases["fine"].dataFileName_)}
scan_ny = {"coarse": 64, "standard": 128, "fine": 256}                   # y cells per level
res = {lvl: level_errors(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  ny    h_y        core err/B_SCALE  order  wall err/B_SCALE  by drift")
prev = None
for lvl in ("coarse", "standard", "fine"):
    core, wall, drift = res[lvl]
    order = "--"
    if prev is not None and prev > 0 and core > 0:
        order = f"{np.log2(prev / core):.2f}"
    print(f"{lvl:<9} {scan_ny[lvl]:<5} {1.0/scan_ny[lvl]:<10.4g} "
          f"{core:<18.3e} {order:<6} {wall:<18.3e} {drift:.1e}")
    prev = core

for lvl in ("coarse", "standard", "fine"):
    assert res[lvl][2] < 1e-10, f"b_y invariant broken at {lvl}: {res[lvl][2]:.1e}"
assert res["standard"][0] < res["coarse"][0], f"no core convergence coarse -> standard: {res}"
assert res["fine"][0] < res["standard"][0], f"no core convergence standard -> fine: {res}"
assert res["standard"][1] < res["coarse"][1] and res["fine"][1] < res["standard"][1], \
    f"wall-layer deviation not convergent: {res}"
print("OK: induced-field error decreases with y-refinement at every level;")
print("    observed order ~1 = the documented wall-cell gradient bias (first order).")

## 3D checks


In [ ]:
# 1) the applied transverse field must remain exactly uniform (exact invariant)
t, by = read_son(case_3d.dataFileName_, "SONDE_BY")
drift_by = float(np.max(np.abs(by - B0)))
print(f"b_y: b(0)={by[0]:.6f}  max|b_y - B0| = {drift_by:.2e}")
assert drift_by < 1e-10, f"applied transverse field drifted: {drift_by:.3e}"
print("OK: b_y = B0 is an exact invariant (S_y = 0).")


In [ ]:
# 2) the core induced field must match the analytic mean-free Hartmann profile
TOL_CORE = 2.5e-2    # max |b_x_sim - b_x_ref| / B_SCALE over the duct core
STATIONARY = 1e-3    # residual transient on the last recorded step

ys, sim, ref, errs, steps = [], [], [], [], []
for tag, y in CORE:
    t, bx = read_son(case_3d.dataFileName_, tag)
    r = b_x_steady(y)
    e = abs(bx[-1] - r) / B_SCALE
    step = abs(bx[-1] - bx[-2]) / B_SCALE
    ys.append(y); sim.append(bx[-1]); ref.append(r); errs.append(e); steps.append(step)
    print(f"y={y:+.7f}  b_x(t_end)={bx[-1]:+.6f}  analytic={r:+.6f}  "
          f"err/B_SCALE={e:.2e}  last-step={step:.2e}")

err, last_step = max(errs), max(steps)
print(f"core: max |b_x - analytic| / B_SCALE = {err:.3e}   (last-step {last_step:.2e})")
assert err < TOL_CORE, f"core induced field off by {err:.3e} (> {TOL_CORE:g})"
assert last_step < STATIONARY, f"not converged to steady state: last step {last_step:.3e}"
print("OK: core induced field matches the analytic insulating-wall Hartmann profile.")


In [ ]:
# 3) the Hartmann-layer deviation is ODD and bounded (known boundary limitation)
TOL_WALL = 1.0e-1

wall_vals, wall_err = [], []
for tag, y in WALL:
    t, bx = read_son(case_3d.dataFileName_, tag)
    r = b_x_steady(y)
    e = abs(bx[-1] - r) / B_SCALE
    wall_vals.append(bx[-1]); wall_err.append(e)
    print(f"y={y:+.7f}  b_x(t_end)={bx[-1]:+.6f}  analytic={r:+.6f}  err/B_SCALE={e:.2e}")

asym = abs(wall_vals[0] + wall_vals[1]) / B_SCALE
print(f"wall-layer: max err/B_SCALE = {max(wall_err):.2e}, odd symmetry |b(-y)+b(y)|/B_SCALE = {asym:.2e}")
assert asym < 1e-10, f"induced field lost its odd symmetry: {asym:.3e}"
assert max(wall_err) < TOL_WALL, f"wall-layer deviation {max(wall_err):.3e} exceeds the documented bound {TOL_WALL:g}"
print("OK: layer deviation is odd-symmetric and within the documented bound.")


In [ ]:
import matplotlib.pyplot as plt

yy = np.linspace(-H, H, 201)
plt.figure(figsize=(6, 4))
plt.plot(yy, [b_x_steady(y) for y in yy], "k-", label="analytic $b_x(y)$ (steady)")
plt.plot([y for _, y in CORE], sim, "o", label="simulated (core probes)")
plt.plot([y for _, y in WALL], wall_vals, "s", mfc="none",
         label="simulated (layer probes, known limitation)")
plt.axhline(0.0, color="grey", lw=0.5)
plt.xlabel("y [m]"); plt.ylabel("$b_x$ [T]")
plt.title("Hartmann 1D: induced field, insulating walls")
plt.legend(); plt.grid(True); plt.show()


## Known limitation: the velocity gradient at wall cells

The M1 source needs the cell gradient $\nabla u$ of the frozen velocity. The
kernel builds it (`Discret_Thyd::grad_u` → `grad_Champ_Face_PolyMAC_MPFA`) by
*averaging the face gradients* (`update_ge()`), and a boundary face only
receives a Dirichlet contribution — the frozen velocity field carries no
hydrodynamic wall condition here. The two cells adjacent to a wall therefore
receive a biased $\nabla u$. Isolating the source (run the case with $\eta=0$
and a velocity $u = a\,y$, which is exactly representable) makes the effect
readable: the implied $\mathrm{d}u/\mathrm{d}y$ is $0.150$ instead of $0.300$ at
the wall cells, and correct ($0.300$) in the interior.

Consequences, measured on this dataset (64 cells across the gap):

- the deviation $b_x^{sim} - b_x^{analytic}$ is an almost exact **linear mode
  in $y$** ($\approx 0.305\,y$), i.e. it vanishes at the centre and is largest
  at the walls: $\approx 1\%$ at $|y| = 0.05$, $\approx 9\%$ at $|y| = 0.49$;
- it is **convergent, first order** (halves when the mesh is refined);
- it is odd (the wall-cell bias is symmetric), so the mean and the odd
  symmetry of the profile are preserved exactly.

This is why the core probes are asserted tightly (a few $10^{-2}$ on a 64-cell
mesh) and the layer probes only against the bound $10^{-1}$: the bound is a
**regression guard on the known limitation**, and it can be tightened once the
boundary treatment of the frozen velocity's gradient is fixed.


## 3D mesh convergence

The shared refinement ladder, adapted to the 1D channel physics: only the
active `y` direction is refined (`coarse` = `5×65×5`, `standard` = y cells ×2
(`5×129×5`), `fine` = y cells ×4 (`5×257×5`)).

Metric: the core-probe error against the analytic steady Hartmann profile
$b_x(y)$, evaluated at each probe's **effective** (cell-centre) position
read from the `.son` header; the observed order is
$\log_2(e_k/e_{k+1})$. The wall-adjacent probes are excluded from the scan
(documented O(h) Green–Gauss wall bias; covered by the baseline checks),
and the exact invariant $b_y = B_0$ is asserted at every level.

In [ ]:
CORE_SCAN = [tag for tag, _ in CORE]     # analytic y is taken from the .son header

def probe_xy(path):
    """Effective (x, y) of a moved point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2])
    raise RuntimeError(f"no probe position header in {path}")

def read_son_dir(subdir, case_name, name):
    """(time, values, (x_eff, y_eff)) of probe `name` in a build subdir."""
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float), probe_xy(path)

def core_error(subdir, case_name):
    """Max |b_x(y_eff) - b_x_steady(y_eff)| / B_SCALE over the core probes."""
    worst = 0.0
    for tag in CORE_SCAN:
        _, bx, (_, y_eff) = read_son_dir(subdir, case_name, tag)
        worst = max(worst, abs(bx[-1] - b_x_steady(y_eff)) / B_SCALE)
    return worst

def by_drift(subdir, case_name):
    """max |b_y(t) - B0| over time at the applied-field probe."""
    _, by = read_son_dir(subdir, case_name, "SONDE_BY")[:2]
    return float(np.max(np.abs(by - B0)))

scan = {"coarse": ("3D/coarse", case_3d.dataFileName_),
        "standard": ("3D/standard", cases_3d["standard"].dataFileName_),
        "fine": ("3D/fine", cases_3d["fine"].dataFileName_)}
scan_nodes = {"coarse": (5, 65, 5), "standard": (5, 129, 5), "fine": (5, 257, 5)}
scan_err = {lvl: core_error(sub, cn) for lvl, (sub, cn) in scan.items()}
scan_drift = {lvl: by_drift(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes     core err/B_SCALE   observed order   by drift")
prev = None
for lvl in ("coarse", "standard", "fine"):
    nx, ny, nz = scan_nodes[lvl]
    e = scan_err[lvl]
    order = "--"
    if prev is not None and prev > 0 and e > 0:
        order = f"{np.log2(prev / e):.2f}"
    print(f"{lvl:<9} {nx:>2}x{ny}x{nz:<2} {e:<18.4e} {order:<16} {scan_drift[lvl]:.2e}")
    assert scan_drift[lvl] < 1e-12, f"applied field drift at {lvl}: {scan_drift[lvl]}"
    prev = e

assert scan_err["standard"] < scan_err["coarse"], f"no convergence coarse -> standard: {scan_err}"
assert scan_err["fine"] < scan_err["standard"], f"no convergence standard -> fine: {scan_err}"
print("OK: core profile error decreases monotonically; b_y = B0 invariant at every level.")

## Computer Performance


In [ ]:
run.tablePerf()
